In [1]:
from utils import prepare_dataframe, merge_csv_files_in_directory

#merge_csv_files_in_directory()
df = prepare_dataframe("merged.csv")
resulting_label_count = df['labels'].nunique()
df


labels
bug            17934
enhancement    16113
clean          15336
performance     5332
refactor        3320
Name: count, dtype: int64
labels
enhancement    15300
bug            15300
clean          15300
Name: count, dtype: int64


,full_repo_name,labels,patch,index
20575,apache/incubator-nemo,enhancement,"@@ -68,4 +68,9 @@ public Set<ExecutorRepresent...",20575
22175,apache/maven-invoker-plugin,enhancement,"@@ -43,12 +43,12 @@\n import java.util.List;\n...",22175
45510,apache/pulsar,enhancement,"@@ -486,8 +486,11 @@ public void testCardinali...",45510
15849,apache/hbase,enhancement,"@@ -1962,9 +1962,9 @@ private void startServic...",15849
5140,apache/cloudstack,enhancement,"@@ -1887,18 +1887,10 @@ public static HostResp...",5140
...,...,...,...,...
24020,apache/netbeans,clean,"@@ -45,7 +45,7 @@ public class WLTargetModuleI...",24020
25558,apache/netbeans,clean,"@@ -95,11 +95,10 @@ public Item getObjectAt(St...",25558
4503,apache/cloudstack,clean,"@@ -113,7 +113,7 @@ public Date getRemoved() {...",4503
27264,apache/netbeans,clean,"@@ -38,7 +38,7 @@ public class SuperPatternFil...",27264


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda:0")

c:\Users\FSM\.conda\envs\code1002\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\FSM\.conda\envs\code1002\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\FSM\.cache\huggingface\hub\models--microsoft--graphcodebert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
import os

embedding_directory = 'embeddings/GCB_Pooler_embeddings_patch_before_largewithcodedata_15300limit3label'

os.makedirs(embedding_directory, exist_ok=True)

In [4]:
from utils import extract_code_from_patch

# Tokenizing using subtracting full code before and after embeddings

# def generate_and_save_embeddings_batch(contents_before, contents_after, indices):
def generate_and_save_embedding(patch, index):

    patch_before, patch_after = extract_code_from_patch(patch)
    
    
    # Tokenize the before and after code patch snippet
    tokens_before = tokenizer(patch_before, truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Tokenize the before and after code snippets
    # tokens_before = tokenizer(list(contents_before), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")
    # tokens_after = tokenizer(list(contents_after), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Get the embeddings for both tokenized sequences
    with torch.no_grad():

        embedding_before = model(**tokens_before).pooler_output
        # embeddings_after = model(**tokens_after).pooler_output
    
    # Subtract the embeddings
    # subtracted_embeddings = embeddings_after - embeddings_before
    # print(subtracted_embeddings.shape)
    
    # Average the subtracted embeddings across the token dimension
    # subtracted_embeddings = subtracted_embeddings.mean(dim=1)
    
    save_path = os.path.join(embedding_directory, f"embedding_{index}.pt")
    torch.save(embedding_before, save_path)
    

In [5]:
import tqdm

for index,row in tqdm.tqdm_notebook(df.iterrows()):
    generate_and_save_embedding(row['patch'],row['index'])

C:\Users\FSM\AppData\Local\Temp\ipykernel_14924\2418546496.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for index,row in tqdm.tqdm_notebook(df.iterrows()):


0it [00:00, ?it/s]